<a href="https://colab.research.google.com/github/georgegiannakidis/uhart-ms-prep/blob/main/Week1_Day5_Probability_Wireshark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Week 1 — Day 5: Probability, Stats & Wireshark Install
### UHart MS CS Pre-Arrival Prep · George · 7:00–8:30 PM
---
**Tonight's structure:**
- `7:00–7:05` → Linear algebra warmup
- `7:05–7:50` → Probability & statistics foundations for ML
- `7:50–8:15` → Install and run Wireshark — capture your first packet
- `8:15–8:30` → 3 key takeaways + Week 1 self-assessment

> 💡 **Watch first (Stats):** [StatQuest Playlist](https://www.youtube.com/playlist?list=PLblh5JKOoLUIcdlgu78MnlATeyx4cEVeR) — watch: "Normal Distribution Clearly Explained", "Probability is not Likelihood", "Histograms Clearly Explained" (~45 min total)
> 💡 **Watch first (Wireshark):** [Wireshark Tutorial — David Bombal](https://www.youtube.com/watch?v=lb1Dw0elw0Q) (first 30 min covers the basics)

---
## ⏮️ Warmup (7:00–7:05)

In [ ]:
import numpy as np
# Quick linear algebra warmup
A = np.array([[4, 2], [1, 3]])
eigenvals, eigenvecs = np.linalg.eig(A)
print('Eigenvalues:', np.round(eigenvals, 2))
print('Largest eigenvalue:', np.round(eigenvals.max(), 2))
print('Direction of max variance:', np.round(eigenvecs[:, np.argmax(eigenvals)], 3))
print('✅ Linear algebra still loaded!')

---
## 📦 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
print('✅ Ready! scipy.stats is your probability toolkit.')

---
## 1️⃣ Probability Basics

**Probability** = measure of how likely an event is. Range: 0 (impossible) to 1 (certain).

Key formulas you need for CS 569:
- P(A) = number of favorable outcomes / total outcomes
- P(A and B) = P(A) × P(B)  *only if A, B are independent*
- P(A or B)  = P(A) + P(B) - P(A and B)
- **Conditional**: P(A | B) = P(A and B) / P(B)

In [ ]:
# Simulate rolling a die — the empirical approach
np.random.seed(42)
n_rolls = 10000
rolls = np.random.randint(1, 7, n_rolls)

# Empirical probability of rolling a 6
p_six_empirical = np.sum(rolls == 6) / n_rolls
p_six_theory    = 1/6

print(f'P(6) empirical ({n_rolls} rolls): {p_six_empirical:.4f}')
print(f'P(6) theoretical:                {p_six_theory:.4f}')
print(f'Difference: {abs(p_six_empirical - p_six_theory):.4f}')
print()
# As n → ∞, empirical → theoretical (Law of Large Numbers)

In [ ]:
# Conditional probability — critical for understanding Naive Bayes (Week 4) and Bayes theorem
# Security scenario: P(attack | alert)

# Known probabilities
p_alert_given_attack   = 0.95   # sensitivity of IDS
p_alert_given_no_attack = 0.05  # false positive rate
p_attack               = 0.01   # base rate: 1% of traffic is an attack

# Bayes theorem: P(attack | alert) = P(alert | attack) * P(attack) / P(alert)
p_alert = (p_alert_given_attack * p_attack +
           p_alert_given_no_attack * (1 - p_attack))

p_attack_given_alert = (p_alert_given_attack * p_attack) / p_alert

print('=== Bayes Theorem — IDS Alert Analysis ===')
print(f'P(alert | attack)    = {p_alert_given_attack}   (95% detection rate)')
print(f'P(alert | no attack) = {p_alert_given_no_attack}   (5% false positive rate)')
print(f'P(attack)            = {p_attack}   (base rate)')
print()
print(f'P(alert)             = {p_alert:.4f}')
print(f'P(attack | alert)    = {p_attack_given_alert:.4f}  = {p_attack_given_alert*100:.1f}%')
print()
print('⚠️  Even with a 95% detection rate, if attacks are rare,')
print('   most alerts are still FALSE POSITIVES! This is why GRC teams')
print("   always ask 'what is the base rate?' before trusting alert numbers.")

---
## 2️⃣ Distributions — The Language of ML

Every ML algorithm assumes some underlying distribution of data. Knowing these lets you choose the right model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
np.random.seed(42)

# ─── Normal (Gaussian) Distribution ───────────────────────────
ax1 = axes[0]
x = np.linspace(-4, 4, 300)
for mean, std, color, label in [(0, 1, '#5B8DEE', 'μ=0, σ=1'),
                                  (1, 0.5, '#2EB87A', 'μ=1, σ=0.5'),
                                  (0, 2, '#E24B4A', 'μ=0, σ=2')]:
    y = stats.norm.pdf(x, mean, std)
    ax1.plot(x * std + mean if mean != 0 else x, y, color=color, linewidth=2, label=label)
ax1.set_title('Normal Distribution', fontsize=11)
ax1.legend(fontsize=8); ax1.set_ylabel('Density'); ax1.grid(alpha=0.3)

# ─── Binomial Distribution ─────────────────────────────────────
ax2 = axes[1]
n, p = 20, 0.3
k = np.arange(0, n+1)
binom_probs = stats.binom.pmf(k, n, p)
ax2.bar(k, binom_probs, color='#9B7EF5', edgecolor='white', alpha=0.85)
ax2.set_title(f'Binomial (n={n}, p={p})', fontsize=11)
ax2.set_xlabel('Number of successes'); ax2.set_ylabel('Probability')
ax2.grid(axis='y', alpha=0.3)

# ─── Uniform Distribution ─────────────────────────────────────
ax3 = axes[2]
samples_uniform = np.random.uniform(0, 10, 2000)
ax3.hist(samples_uniform, bins=20, color='#E8A020', edgecolor='white', alpha=0.85, density=True)
ax3.set_title('Uniform Distribution [0, 10]', fontsize=11)
ax3.set_xlabel('Value'); ax3.set_ylabel('Density'); ax3.grid(alpha=0.3)

plt.suptitle('Probability Distributions Used in ML', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

---
## 3️⃣ Descriptive Statistics — What Your Data Looks Like

These are the first things you compute before training any ML model.

In [ ]:
# Generate fake CVSS score dataset (like a real vulnerability database)
np.random.seed(0)
cvss_data = np.concatenate([
    np.random.normal(9.2, 0.4, 50),    # critical vulnerabilities
    np.random.normal(7.5, 0.6, 100),   # high
    np.random.normal(5.0, 1.2, 200),   # medium
    np.random.normal(2.5, 0.8, 80),    # low
])
cvss_data = np.clip(cvss_data, 0, 10)

print('=== Descriptive Statistics ===')
print(f'Count:      {len(cvss_data)}')
print(f'Mean:       {np.mean(cvss_data):.3f}')
print(f'Median:     {np.median(cvss_data):.3f}')
print(f'Std dev:    {np.std(cvss_data):.3f}')
print(f'Variance:   {np.var(cvss_data):.3f}')
print(f'Min / Max:  {np.min(cvss_data):.2f} / {np.max(cvss_data):.2f}')
print(f'Quartiles:  {np.percentile(cvss_data, [25, 50, 75]).round(2)}')
print()
print('Mean vs Median:', end=' ')
print('mean > median' if np.mean(cvss_data) > np.median(cvss_data) else 'mean ≤ median')
print('(This tells you if the distribution is skewed!)')

In [ ]:
# Visualize the distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram with KDE overlay
ax1 = axes[0]
ax1.hist(cvss_data, bins=30, color='#5B8DEE', edgecolor='white', alpha=0.7, density=True, label='Data')
kde_x = np.linspace(0, 10, 300)
kde = stats.gaussian_kde(cvss_data)
ax1.plot(kde_x, kde(kde_x), color='#E24B4A', linewidth=2, label='KDE')
ax1.axvline(np.mean(cvss_data), color='#2EB87A', linestyle='--', linewidth=1.5, label=f'Mean={np.mean(cvss_data):.1f}')
ax1.axvline(np.median(cvss_data), color='#E8A020', linestyle='--', linewidth=1.5, label=f'Median={np.median(cvss_data):.1f}')
ax1.set_title('CVSS Score Distribution', fontsize=11)
ax1.set_xlabel('CVSS Score'); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# Box plot
ax2 = axes[1]
bp = ax2.boxplot(cvss_data, vert=True, patch_artist=True,
                  boxprops=dict(facecolor='#5B8DEE', alpha=0.6),
                  medianprops=dict(color='#E24B4A', linewidth=2))
ax2.set_title('CVSS Box Plot', fontsize=11)
ax2.set_ylabel('CVSS Score')
ax2.grid(axis='y', alpha=0.3)
# Annotate
for label, val, pos in [('Q1', np.percentile(cvss_data, 25), 1.05),
                          ('Median', np.median(cvss_data), 1.05),
                          ('Q3', np.percentile(cvss_data, 75), 1.05)]:
    ax2.text(pos, val, f'{label}: {val:.1f}', fontsize=8, va='center', color='#444')

plt.tight_layout(); plt.show()

---
## 4️⃣ Central Limit Theorem — Why Normal Distribution is Everywhere

No matter what the underlying distribution is, the **mean of large samples** will always follow a normal distribution. This justifies using many ML algorithms that assume normality.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
np.random.seed(42)

sample_sizes = [1, 5, 30]
# Original distribution: uniform (very non-normal)
pop = np.random.uniform(0, 10, 100000)

for col, n in enumerate(sample_sizes):
    # Distribution of individual samples (top row)
    axes[0, col].hist(pop[:1000], bins=30, color='#5B8DEE', alpha=0.7, edgecolor='white')
    axes[0, col].set_title(f'Population (Uniform)', fontsize=9)
    axes[0, col].set_ylabel('Count')

    # Distribution of SAMPLE MEANS (bottom row) — this is the CLT!
    sample_means = [np.mean(np.random.choice(pop, n)) for _ in range(2000)]
    axes[1, col].hist(sample_means, bins=30, color='#2EB87A', alpha=0.7, edgecolor='white')
    axes[1, col].set_title(f'Sample Means (n={n}) → {"Normal!" if n>=30 else "Not yet..."}', fontsize=9)
    axes[1, col].set_ylabel('Count of means')

plt.suptitle('Central Limit Theorem — Sample Means Become Normal as n Grows', fontsize=11)
plt.tight_layout(); plt.show()
print('By n=30, the distribution of sample means looks normal regardless of the population shape.')
print('This is why many ML algorithms work well even on non-normal data!')

---
## 5️⃣ Correlation & Covariance

**Correlation** measures the linear relationship between two variables.
- r = 1: perfect positive linear relationship
- r = 0: no linear relationship
- r = -1: perfect negative linear relationship

Critical for: feature selection, understanding which input variables predict the output.

In [ ]:
np.random.seed(55)
n = 100
# Create 3 relationships
x = np.random.randn(n)
y_pos  = x + np.random.randn(n) * 0.5        # strong positive correlation
y_none = np.random.randn(n)                   # no correlation
y_neg  = -x + np.random.randn(n) * 0.5       # strong negative correlation

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, y, title in zip(axes,
                         [y_pos, y_none, y_neg],
                         ['Strong positive', 'No correlation', 'Strong negative']):
    r = np.corrcoef(x, y)[0, 1]
    ax.scatter(x, y, alpha=0.5, color='#5B8DEE', s=25)
    # Regression line
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m*x_line + b, color='#E24B4A', linewidth=2)
    ax.set_title(f'{title}
r = {r:.2f}', fontsize=11)
    ax.grid(alpha=0.3)

plt.suptitle('Pearson Correlation Coefficient', fontsize=12)
plt.tight_layout(); plt.show()

---
## 6️⃣ Wireshark — Your First Packet Capture

**Wireshark** is the most widely used network protocol analyzer. You'll use it for ALL CS 555 labs.

### Install:
1. Go to [wireshark.org/download.html](https://www.wireshark.org/download.html)
2. Download for your OS (Windows/Mac/Linux)
3. Install with default settings

### Your first capture (run this checklist below):

In [ ]:
# Wireshark Quick-Start Checklist
# (run this cell to print your checklist — then do each step in Wireshark)

steps = [
    ('Install', 'Download and install from wireshark.org/download.html'),
    ('Open', 'Launch Wireshark — you should see a list of network interfaces'),
    ('Select', 'Double-click your Wi-Fi or Ethernet interface (the one with traffic)'),
    ('Capture', 'Click the blue shark-fin button (or press Ctrl+E) to start capturing'),
    ('Generate', 'Open your browser and visit http://google.com — this creates packets'),
    ('Stop', 'Click the red square (or press Ctrl+E again) to stop capture after 20 seconds'),
    ('Filter DNS', 'In the filter bar type: dns — press Enter — see DNS queries!'),
    ('Filter HTTP','Change filter to: http — see HTTP requests to google'),
    ('Filter TCP', 'Change filter to: tcp — see the 3-way handshakes (SYN, SYN-ACK, ACK)'),
    ('Save', 'File → Save As → week1_first_capture.pcapng'),
]

print('🦈 WIRESHARK FIRST CAPTURE CHECKLIST')
print('='*55)
for i, (name, desc) in enumerate(steps, 1):
    print(f'{i:2}. [{name:8}] {desc}')

print()
print('🎯 WHAT TO LOOK FOR:')
print('  - In DNS filter: see your computer asking "where is google.com?"')
print('  - In TCP filter: find a SYN packet — right-click → Follow → TCP Stream')
print('  - Notice the 3-way handshake: SYN → SYN-ACK → ACK')
print()
print('📝 Record what you see in the Session Log below!')

---
## 🏋️ Practice Exercises (7:50–8:15 PM)

In [ ]:
# ── Exercise 1 ──────────────────────────────────────────────────
# You're testing an IDS system on a network with 10,000 packets.
# - 100 are actual attacks (base rate = 0.01)
# - IDS sensitivity: 90% (P(alert|attack) = 0.90)
# - IDS false positive rate: 2% (P(alert|no attack) = 0.02)
#
# Using Bayes theorem: What is P(actual attack | IDS alert)?
# This tells you how much to TRUST an alert.

p_attack = 0.01
p_alert_given_attack = 0.90
p_alert_given_no_attack = 0.02

# YOUR CODE HERE
# Expected: surprisingly low — this is the base rate fallacy!

In [ ]:
# ── Exercise 2 ──────────────────────────────────────────────────
# Generate 1000 random CVSS scores from a normal distribution: mean=6.5, std=1.8
# Then compute:
# 1. What % of scores are Critical (>= 9.0)?
# 2. What % are below 4.0 (Low)?
# 3. Plot a histogram with threshold lines

# YOUR CODE HERE

In [ ]:
# ── Exercise 3 ──────────────────────────────────────────────────
# Compute the CORRELATION MATRIX for this vulnerability dataset
# Use np.corrcoef() or pd.DataFrame.corr()
# Then create a heatmap of the correlations using matplotlib imshow()
# Which pair of features is most correlated?

import pandas as pd
vuln_df = pd.DataFrame({
    'CVSS':           [9.8, 7.2, 4.5, 9.1, 6.8, 8.5, 9.3, 3.2, 7.8, 5.5],
    'Days_Exposed':   [45,  20,  10,  60,  30,  25,  50,  5,   35,  15],
    'Attack_Surface': [0.9, 0.6, 0.3, 0.8, 0.5, 0.7, 0.9, 0.2, 0.7, 0.4],
    'Patch_Lag':      [30,  5,   2,   40,  15,  20,  35,  0,   10,  5]
})

# YOUR CODE HERE

In [ ]:
# ── Exercise 4 ──────────────────────────────────────────────────
# Demonstrate the Central Limit Theorem with a HIGHLY skewed distribution
# Use an exponential distribution (very right-skewed)
# Show that sample means (n=50) become normal even from this skewed data

# YOUR CODE HERE
# Use: np.random.exponential(scale=2, size=10000)
# Then: sample 2000 batches of n=50, compute their means, plot the distribution

In [ ]:
# ── Exercise 5 — Full Stats Workflow ──────────────────────────────
# You're a GRC analyst receiving a set of response times (in hours)
# for patching critical vulnerabilities across 5 teams.
#
# 1. Compute: mean, median, std, and 90th percentile for each team
# 2. Which team has the most CONSISTENT patching speed? (hint: lowest std)
# 3. Which team has the highest MEAN time? (most concerning for compliance)
# 4. Plot a box plot comparing all teams

np.random.seed(7)
teams = {
    'Team Alpha': np.random.normal(48, 12, 30),    # mean 48h, std 12
    'Team Beta':  np.random.normal(72, 30, 30),    # mean 72h, std 30
    'Team Gamma': np.random.normal(24, 5,  30),    # mean 24h, std 5
    'Team Delta': np.random.normal(96, 20, 30),    # mean 96h, std 20
    'Team Echo':  np.random.normal(36, 8,  30),    # mean 36h, std 8
}

# YOUR CODE HERE

---
## 🏆 Week 1 Self-Assessment (8:15–8:30 PM)

Rate your confidence for each topic before you close the notebook.
**Be honest — this tells you what to review on Saturday.**

In [ ]:
# Run this self-assessment after completing all exercises
topics = {
    'NumPy: arrays, slicing, broadcasting': None,   # 'solid' | 'getting_it' | 'shaky'
    'NumPy: dot product, matrix multiply': None,
    'Pandas: DataFrames, filtering, groupby': None,
    'Pandas: handling missing data': None,
    'Matplotlib: line, bar, histogram, scatter': None,
    'Scikit-learn: fit/transform/predict pattern': None,
    'Linear algebra: vectors and dot product': None,
    'Linear algebra: matrix multiplication': None,
    'Linear algebra: eigenvalues (concept)': None,
    'Probability: Bayes theorem': None,
    'Statistics: mean, std, distributions': None,
    'Wireshark: captured first packet': None,
}

icons = {'solid': '✅', 'getting_it': '🟡', 'shaky': '🔁', None: '⬜'}

print('='*52)
print('WEEK 1 SELF-ASSESSMENT — Pre-Arrival Prep')
print('='*52)
for topic, status in topics.items():
    icon = icons.get(status, '⬜')
    print(f'{icon} {topic}')

shaky = [t for t, s in topics.items() if s == 'shaky']
if shaky:
    print(f'\n⚠️  REVIEW ON SATURDAY: {len(shaky)} topics need attention')
    for t in shaky:
        print(f'   → {t}')
else:
    print('\n🎉 All assessed! Check back Saturday for targeted review.')

---
## 🎉 Week 1 Complete!

### Your Saturday review plan:
1. Re-run the self-assessment above with honest ratings
2. Open notebooks for any 'shaky' topics and redo exercises without looking at answers
3. Wipe your Wireshark capture and recapture — this time try to identify the DNS handshake and TCP stream manually
4. Commit all 5 notebooks to GitHub — you have real work to show!

### Week 2 starts Monday: CS 555 — Kurose & Ross Chapter 1
Watch: [Kurose's video lectures](https://gaia.cs.umass.edu/kurose_ross/videos/1/) before Monday's session.